# Run BEHAV3D feature extraction and T cell analysis with Cellpose

**Expected duration full processing per full time-series BEHAV3D position: ~4-5 hours**

This notebook runs the feature extraction and behavioral analysis pipeline of BEHAV3D with Cellpose segmentation

To work correctly, it requires the following input:
- A metadata ***.csv*** containing information on every sample to be processed
- A 5D (tczyx) microscopy image (***.czi/.tiff/.zarr***)

*Optional*: You can supply your own segmentation/tracking in the metadata .csv and skip the segmentation/tracking by BEHAV3D

### Load required packages

In [1]:
#Load the packages
from behav3d.preprocessing.segmentation.cellpose_prediction import run_cellpose_prediction,run_cellpose_segmentation,run_otsu_threshold_segmentation_from_zarr,visualize_cellpose_sample

from behav3d.utils import load_behav3d_metadata, check_behav3d_metadata
from behav3d.utils.behav3d_widgets import (
    PathPicker, 
    MetadataLoader, 
    convert_zarr_button, 
    DimOrderTable, 
    TrackingPanel, 
    TrackingVisualizationPanel,
    FeatureExtractionPanel,
    TcellFilterPanel,
    TCellAnalysisPanel,
    OrganoidFilterPanel,
    OrganoidAnalysisPanel,
    BackprojectionPanel
)
import pandas as pd
from pathlib import Path

import torch
import zarr
import os

import ipywidgets as widgets
from IPython.display import display, clear_output

import random
random.seed(42)  # sets the seed for the random module

import numpy as np
np.random.seed(42)  # sets the seed for numpy random functions


%matplotlib inline

In [2]:
# ---- App-wide state (lives in the kernel across cells) ----
import traitlets
from traitlets import Bool, Any

class _AppState(traitlets.HasTraits):
    # signals
    metadata_ready  = Bool(False)   # flips True when metadata is loaded
    segmentation_ready = Bool(False)
    tracking_ready  = Bool(False)
    # shared objects
    mdl     = Any(None)             # your MetadataLoader instance
    metadata= Any(None)

# reuse the same instance if you re-run this cell
APP = globals().get("APP") or _AppState()
globals()["APP"] = APP

# ---- tiny utility: run builder now or when a signal turns True ----
import ipywidgets as widgets

def render_when(container: widgets.Box, signal_name: str, builder):
    """
    If APP.<signal_name> is True, render immediately.
    Otherwise, register a one-shot observer that renders when it becomes True.
    `builder()` must return a widget (or a tuple/list of widgets).
    """
    # fast path: already ready
    if bool(getattr(APP, signal_name)):
        built = builder()
        container.children = built if isinstance(built, (tuple, list)) else (built,)
        return

    # otherwise, wait for the flip
    def _on(change):
        if change["name"] == signal_name and change["new"] is True:
            APP.unobserve(_on, names=[signal_name])  # one-shot
            built = builder()
            container.children = built if isinstance(built, (tuple, list)) else (built,)

    APP.observe(_on, names=[signal_name])

### Set BEHAV3D parameters

***NOTE*** 

Dynamic Time Warping seems to work best if all tracks are of equal length. To get tracks to equal length, set both ***tcell_min_track_length*** and ***tcell_max_track_length*** to the same value

**!!! Make sure channel indexing in metadata csv starts at channel 0!!!**

Checks the supplied metadata .csv for:
- Are all required columns filled in?
- Do all the supplied paths per row exist?

In [3]:
# Import metadata
output_dir_picker = PathPicker(
    mode='dir',
    start_dir='.',
    default=r"runs/TKI",
    description='Output Dir:',
)

metadata_path_picker = PathPicker(
    mode='file',
    start_dir='.',
    default=r"runs/TKI/metadata.csv",
    description='Metadata .csv:',
    filter_pattern='*.csv'  # optional
)

metadata_loader = MetadataLoader(
    metadata_path_picker=metadata_path_picker, 
    output_dir_picker=output_dir_picker,
    button_description='Load Metadata'
    )
display(metadata_loader)

# wire the loader into APP
APP.mdl = metadata_loader

# Flip the signal when load() returns successfully
_orig_load = metadata_loader.load
def _wrapped_load(*a, **k):
    try:
        return _orig_load(*a, **k)
    finally:
        if getattr(metadata_loader, "metadata", None) is not None:
            APP.metadata = metadata_loader.metadata
            APP.metadata_ready = True
metadata_loader.load = _wrapped_load

# (Optional backup: if _busy turns False and metadata exists, also flip)
def _maybe_ready(change):
    if change["name"] == "_busy" and change["new"] is False and getattr(metadata_loader, "metadata", None) is not None:
        APP.metadata = metadata_loader.metadata
        APP.metadata_ready = True
metadata_loader.observe(_maybe_ready, names="_busy")

print("Click ‘Load metadata’. Other cells will update automatically.")


MetadataLoader(children=(PathPicker(children=(HBox(children=(Textarea(value='runs/TKI', description='Output Di…

Click ‘Load metadata’. Other cells will update automatically.


In [5]:
# ---------- Run-ALL controller (shows only after metadata is ready) ----------
import ipywidgets as widgets
from IPython.display import display

# Global registry (create if not already defined)
try:
    RUN_ALL_REGISTRY
except NameError:
    RUN_ALL_REGISTRY = []   # list of (name, callable)

# If you already defined register_runnable earlier, keep it.
# Otherwise define it here (and auto-refresh the status label if present).
def register_runnable(obj, name: str = None):
    import ipywidgets as widgets
    if isinstance(obj, widgets.Button):
        label = name or (obj.description or "Button")
        RUN_ALL_REGISTRY.append((label, obj.click))
    elif callable(obj):
        label = name or getattr(obj, "__name__", "step")
        RUN_ALL_REGISTRY.append((label, obj))
    else:
        raise TypeError("register_runnable expects a Button or a callable")
    # refresh status label if controller is already built
    if "_run_all_status_lbl" in globals():
        names = [n for n, _ in RUN_ALL_REGISTRY]
        _run_all_status_lbl.value = f"{len(names)} steps registered: " + (", ".join(names) if names else "—")

# Optional helper to find & register buttons inside composite UIs
_REGISTERED_BUTTONS = globals().get("_REGISTERED_BUTTONS", set())
def iter_buttons(widget):
    if isinstance(widget, widgets.Button):
        yield widget
    if hasattr(widget, "children"):
        for ch in widget.children:
            yield from iter_buttons(ch)
    if isinstance(widget, (list, tuple)):
        for w in widget:
            yield from iter_buttons(w)
    if isinstance(widget, dict):
        for w in widget.values():
            yield from iter_buttons(w)

def register_buttons_in(widget, prefix: str = ""):
    for btn in iter_buttons(widget):
        if btn in _REGISTERED_BUTTONS:
            continue
        _REGISTERED_BUTTONS.add(btn)
        register_runnable(btn, prefix + (btn.description or "Button"))

# Placeholder that shows until metadata is ready
run_all_container = widgets.VBox([widgets.HTML("<i>Waiting for metadata…</i>")])
display(run_all_container)

def build_run_all():
    global _run_all_status_lbl  # so register_runnable can update it
    run_all_btn = widgets.Button(description="Run ALL", button_style="warning", icon="fast-forward",
                                 layout=widgets.Layout(width="160px"))
    _run_all_status_lbl = widgets.HTML("0 steps registered")
    log_out = widgets.Output()

    def _refresh_status():
        names = [n for n, _ in RUN_ALL_REGISTRY]
        _run_all_status_lbl.value = f"{len(names)} steps registered: " + (", ".join(names) if names else "—")

    def _on_run_all(_):
        log_out.clear_output()
        with log_out:
            for name, action in RUN_ALL_REGISTRY:
                print(f"▶ {name}")
                try:
                    action()   # blocks until the step finishes
                except Exception as e:
                    import traceback
                    print(f"   ⚠️ {name} failed: {e}")
                    traceback.print_exc()

    run_all_btn.on_click(_on_run_all)
    _refresh_status()

    # Return the controller UI
    return widgets.VBox([
        widgets.HBox([run_all_btn, _run_all_status_lbl],
                     layout=widgets.Layout(gap="10px", align_items="center")),
        log_out
    ])

# Only build/show the controller once APP signals metadata is ready:
render_when(run_all_container, "metadata_ready", build_run_all)

In [6]:
# Panel for supplying dimension order and converting input image to zarr for multiprocessing and memory efficiency

container = widgets.VBox([widgets.HTML("<i>Waiting for metadata…</i>")])
display(container)

def build_dim_and_convert():
    dim = DimOrderTable(metadata_loader=APP.mdl, auto_write=True)
    convert_ui = convert_zarr_button(metadata_loader=APP.mdl, dim_order_widget=dim)
    register_buttons_in(convert_ui, prefix="")
    return [dim.widget, convert_ui]  # both rendered together

render_when(container, "metadata_ready", build_dim_and_convert)


---

# Run Cellpose segmentation

---

### Segment all input images
*Expected duration: ~3.5 hours per image, ~20 seconds per timepoint*

#### Load the Cellpose models

In [7]:
# Choose Cellpose models
cellpose_org_widget = PathPicker(
    mode='file',
    start_dir='.',
    description='Organoid Cellpose model path:',
    default=r"models/cellpose_organoids__channel0-organoid",
    description_width='auto'
)

cellpose_cell_widget = PathPicker(
    mode='file',
    start_dir='.',
    description='ImmuneCell Cellpose model path:',
    default=r"models/cellpose_tcells__channel0-tcell",
    description_width='auto'
)

load_button = widgets.Button(description="Load Organoid and ImmuneCell Cellpose models", button_style='success',layout={'width': 'max-content'})
output_area = widgets.Output()

def on_load_clicked(b):
    global pretrained_model_dir_org, pretrained_model_dir_cell, org_channel_order, cell_channel_order, labels
    pretrained_model_dir_org = cellpose_org_widget.value
    pretrained_model_dir_cell = cellpose_cell_widget.value
    org_channel_order = pretrained_model_dir_org.split(os.path.sep)[-1].split('__')[-1].split('_')
    org_channel_order = [ch.split('-')[-1] for ch in org_channel_order]
    cell_channel_order = pretrained_model_dir_cell.split(os.path.sep)[-1].split('__')[-1].split('_')
    cell_channel_order = [ch.split('-')[-1] for ch in cell_channel_order]
    labels=pd.unique(metadata_loader.metadata[[column for column in metadata_loader.metadata.columns if 'channel' in column and 'number' not in column]].values.flatten())

    with output_area:
        clear_output()
        print(f"✅ Loaded Cellpose model paths")
        print(f"  - Path for the Organoid model:  {pretrained_model_dir_org}")
        print(f"  - Channels with whom the Organoid model has been trained:    {org_channel_order}")
        print(f"  - Path for the ImmuneCell model:    {pretrained_model_dir_cell}")
        print(f"  - Channels with whom the ImmuneCell model has been trained:    {cell_channel_order}")

load_button.on_click(on_load_clicked)

# Display both widgets + button + output
display(cellpose_org_widget, cellpose_cell_widget, load_button, output_area)

PathPicker(children=(HBox(children=(Textarea(value='models/cellpose_organoids__channel0-organoid', description…

PathPicker(children=(HBox(children=(Textarea(value='models/cellpose_tcells__channel0-tcell', description='Immu…

Button(button_style='success', description='Load Organoid and ImmuneCell Cellpose models', layout=Layout(width…

Output()

In [8]:
# Selector for inference on organoids
label_to_segment_org_widget=widgets.Dropdown(
    options=labels,
    value=labels[0],
    description='Which label do you want to segment with the Organoid model: ',
    style={'description_width': 'auto'},
    layout={'width': 'max-content'},
)
selector_org_widget=[]
for i in range(len(org_channel_order)):
    selector_org_widget.append(widgets.Dropdown(
    options=labels,
    value=labels[0],
    description='Which label do you want to put on channel '+str(i)+'?',
    style={'description_width': 'auto'},
    layout={'width': 'max-content'},
))

load_button = widgets.Button(description="Apply the Organoid Cellpose model", button_style='success',layout={'width': 'max-content'})
output_area = widgets.Output()

def on_load_clicked(b):
    global label_to_segment_org, input_channels_org
    label_to_segment_org = label_to_segment_org_widget.value
    input_channels_org = [selector_org_widget[i].value for i in range(len(selector_org_widget))]

    with output_area:
        clear_output()
        # Run organoid segmentation with Cellpose    
        metadata = run_cellpose_segmentation(
            output_dir          = output_dir_picker.value,
            metadata            = metadata_loader.metadata,
            pretrained_model_dir= pretrained_model_dir_org,
            input_channels      = input_channels_org,
            timepoint_range     = (0, 2),      # None → all frames
            label_name          = label_to_segment_org,    
            label_type          = "organoid",    
            overwrite           = True,
        )
        # Persist the new mask paths back to disk so later stages can use them
        metadata_loader.metadata.to_csv(metadata_loader.file_picker.value, index=False)
        print("✓ Organoid segmentation finished and metadata.csv updated")

load_button.on_click(on_load_clicked)
box= widgets.VBox([label_to_segment_org_widget, *selector_org_widget,load_button, output_area],
                             layout=widgets.Layout(grid_gap = '20px 25px'))
organoid_tab = widgets.Tab(children=[box], layout={'width': 'max-content'})
organoid_tab.set_title(0, 'Channel config (org)')
display(organoid_tab)

In [11]:
# Selector for inference on T cells
label_to_segment_cell_widget=widgets.Dropdown(
    options=labels,
    value=labels[0],
    description='Which label do you want to segment with the ImmuneCell model?',
    style={'description_width': 'auto'},
    layout={'width': 'max-content'},
)
selector_cell_widget=[]
for i in range(len(cell_channel_order)):
    selector_cell_widget.append(widgets.Dropdown(
    options=labels,
    value=labels[0],
    description='Which label do you want to put on channel '+str(i)+'?',
    style={'description_width': 'auto'},
    layout={'width': 'max-content'},
))

load_button = widgets.Button(description="Apply the ImmuneCell Cellpose model", button_style='success',layout={'width': 'max-content'})
output_area = widgets.Output()

def on_load_clicked(b):
    global label_to_segment_cell, input_channels_cell
    label_to_segment_cell = label_to_segment_cell_widget.value
    input_channels_cell = [selector_cell_widget[i].value for i in range(len(selector_cell_widget))]
    
    with output_area:
        clear_output()
        # Run organoid segmentation with Cellpose    
        metadata = run_cellpose_segmentation(
            output_dir          = output_dir_picker.value,
            metadata            = metadata_loader.metadata,
            pretrained_model_dir= pretrained_model_dir_cell,
            input_channels      = input_channels_cell,
            timepoint_range     = (0, 2),      # None → all frames
            label_name          = label_to_segment_cell,    
            label_type          = "tcell",    
            overwrite           = True,
        )
        # Persist the new mask paths back to disk so later stages can use them
        metadata_loader.metadata.to_csv(metadata_loader.file_picker.value, index=False)
        print("✓ Immune cell segmentation finished and metadata.csv updated")

load_button.on_click(on_load_clicked)
box= widgets.VBox([label_to_segment_cell_widget, *selector_cell_widget, load_button, output_area],
                             layout=widgets.Layout(grid_gap = '20px 25px'))
tcell_tab = widgets.Tab(children=[box], layout={'width': 'max-content'})
tcell_tab.set_title(0, 'Channel config (cell)')
display(tcell_tab)

In [17]:
# Run dead dye segmentation
run_otsu_threshold_segmentation_from_zarr(
    output_dir          = output_dir_picker.value,
    metadata            = metadata_loader.metadata,
    mask_suffix         = "_mask_dead",
    timepoint_range     = (0, 2), 
    overwrite           = False,
)


Processing r06c05f03_timeseries_c...
[INFO] Global Otsu threshold: 0.06878246366977692
[SKIP] Mask already exists: runs/TKI/images/r06c05f03_timeseries_c/r06c05f03_timeseries_c_mask_dead.zarr

Processing A5_F1...
[INFO] Global Otsu threshold: 0.115234375
[SKIP] Mask already exists: runs/TKI/images/A5_F1/A5_F1_mask_dead.zarr

Processing A8_F1...
[INFO] Global Otsu threshold: 0.09623226523399353
[SKIP] Mask already exists: runs/TKI/images/A8_F1/A8_F1_mask_dead.zarr

Processing C10_F1...
[INFO] Global Otsu threshold: 0.09275908768177032
[SKIP] Mask already exists: runs/TKI/images/C10_F1/C10_F1_mask_dead.zarr

Processing G3_F1...
[INFO] Global Otsu threshold: 0.08876653015613556
[SKIP] Mask already exists: runs/TKI/images/G3_F1/G3_F1_mask_dead.zarr


In [12]:
sample_dropdown_widget = widgets.Dropdown(
    options=metadata_loader.metadata.loc[:,'sample_name'],
    value=metadata_loader.metadata.loc[0,'sample_name'],
    description="Sample:",
    layout=widgets.Layout(width="350px"),
    disabled=False,
)

display(sample_dropdown_widget)

load_button = widgets.Button(description="Visualize the sample", button_style='success',layout={'width': 'max-content'})
output_area = widgets.Output()

def on_load_clicked(b):
    global channel_colors
    channel_colors=[]
    with output_area:
        clear_output()
        sample_selected=metadata_loader.metadata.loc[metadata_loader.metadata['sample_name']==sample_dropdown_widget.value,:]
        labels_sample=sample_selected[[column for column in sample_selected.columns if 'channel' in column and 'number' not in column]].values.flatten()
        for i in range(len(labels_sample)):
            if (labels_sample[i]=='tcell'):
                channel_colors.append('cyan')
            elif (labels_sample[i]=='organoid'):
                channel_colors.append('yellow')
            elif (labels_sample[i]=='death'):
                channel_colors.append('red')
            elif (labels_sample[i]=='macrophage'):
                channel_colors.append('green')
        print(channel_colors)  
        # Visualise the first sample interactively in Napari
        visualize_cellpose_sample(
            output_dir      = output_dir_picker.value,
            sample_name     = sample_dropdown_widget.value,   # choose any row
            timepoint_range = (0, 2),                   # optional
            channel_colors  = channel_colors
        )

load_button.on_click(on_load_clicked)
display(load_button, output_area)


Dropdown(description='Sample:', layout=Layout(width='350px'), options=('r06c05f03_timeseries_c', 'A5_F1', 'A8_…

Button(button_style='success', description='Visualize the sample', layout=Layout(width='max-content'), style=B…

Output()

### Track all T cells images & Organoid

In [ ]:
# Panel for T cell tracking

container = widgets.VBox([widgets.HTML("<i>Waiting for metadata…</i>")])
display(container)

def build_tracking_panel():
    tracking_panel = TrackingPanel(
        metadata_loader=metadata_loader, 
        cell_type='tcell'
        )
    register_buttons_in(tracking_panel.btn_run, prefix="")
    return tracking_panel.ui  # both rendered together

render_when(container, "metadata_ready", build_tracking_panel)


In [ ]:
#Pannel for organoid tracking

container = widgets.VBox([widgets.HTML("<i>Waiting for metadata…</i>")])
display(container)

def build_tracking_panel():
    tracking_panel = TrackingPanel(
        metadata_loader=metadata_loader, 
        cell_type='organoid'
        )
    register_buttons_in(tracking_panel.btn_run, prefix="")
    return tracking_panel.ui  # both rendered together

render_when(container, "metadata_ready", build_tracking_panel)


In [ ]:
# Panel for visualizing tracking results
container = widgets.VBox([widgets.HTML("<i>Waiting for metadata…</i>")])
display(container)

def build_tracking_visualization_panel():
    return TrackingVisualizationPanel(metadata_loader=metadata_loader, channel_colors=channel_colors)._panel

render_when(container, "metadata_ready", build_tracking_visualization_panel)

---

# Calculate the track features, filter tracks and summarize track features

---

**calculate_track_features** <br>
Calculates movement, contact and intensity features for each timepoint in all tracks per experiment

**filter_tracks** <br>
Filter out tracks based on:
- Maximum experiment length (tcell_exp_duration)    
- Minimum track length (tcell_min_track_length)
- Tracks starting at timepoint 1 with a dead dye mean over the dead_dye_threshold (dead_dye_threshold)

Additonally, all tracks are cut down to:
- Maximum track length (tcell_max_track_length)

**summarize_track_features** <br>
Summarizes the features into one value for each TrackID per experiment

--------------------


## T cells

**Calculation of track features**

*Expected duration: ~30 minutes*

In [ ]:
# T cell feature extraction

container = widgets.VBox([widgets.HTML("<i>Waiting for metadata…</i>")])
display(container)

def build_feature_extraction_panel():
    feature_extraction_panel = FeatureExtractionPanel(
        metadata_loader=metadata_loader, 
        cell_type='tcell'
        )
    register_buttons_in(feature_extraction_panel.btn_run, prefix="")
    return feature_extraction_panel.ui  # both rendered together

render_when(container, "metadata_ready", build_feature_extraction_panel)

## Organoids

In [ ]:
# Organoid feature extraction

container = widgets.VBox([widgets.HTML("<i>Waiting for metadata…</i>")])
display(container)

def build_feature_extraction_panel():
    feature_extraction_panel = FeatureExtractionPanel(
        metadata_loader=metadata_loader, 
        cell_type='organoid'
        )
    register_buttons_in(feature_extraction_panel.btn_run, prefix="")
    return feature_extraction_panel.ui  # both rendered together

render_when(container, "metadata_ready", build_feature_extraction_panel)

---

# Perform data analysis

---

### Filtering Parameters 

- **exp_duration**  
  Maximum length of the experiment in timepoints.  
  All data points after this timepoint will be excluded.  
  *Use this to remove late timepoints where imaging quality may degrade significantly.*

- **tcell_min_track_length**  
  Minimum track length to include in the analysis.  
  *Tracks shorter than this are filtered out, typically removing segmentation or tracking errors.*

- **tcell_max_track_length**  
  Maximum track length to include in the analysis.  
  *Longer tracks will be truncated to this length to ensure comparability across tracks of equal length.*


## T cell behavioral analysis

**Filtering tracks and summarizing features**

*Expected duration: <1 minute*

In [ ]:
#Panel to filter T cell tracks

container = widgets.VBox([widgets.HTML("<i>Waiting for metadata…</i>")])
display(container)

def build_track_filter_panel():
    tcell_track_filter = TcellFilterPanel(
        metadata_loader=metadata_loader, 
        cell_type='tcell'
        )
    register_buttons_in(tcell_track_filter.btn_run, prefix="")
    return tcell_track_filter.ui  # both rendered together

render_when(container, "metadata_ready", build_track_filter_panel)

***Dynamic Time Warping*** <br>

Performs Dynamic Time Warping on the various tracks using various features to calculate a distance matrix between all tracks

Select the features to use for analysis

Features used for dynamic time warping by default:
- z_mean_square_displacement
- z_speed
- z_mean_dead_dye
- tcell_contact
- organoid_contact



*Expected duration: ~1 minute*


***UMAP fitting and clustering*** <br>

This distance matrix is fitted into a UMAP, and K-means clustering is used to cluster the tracks into clusters

Parameters for UMAP and clustering can be adjusted:
- umap_minimal_distance
- umap_n_neighbors
- nr_of_clusters

***Cluster features display*** <br>

The results of the clustering are displayed in multiple ways:
- The UMAP with clusters and features backprojected onto this UMAP (BEHAV3D_UMAP_clusters.csv)
- A heatmap containing the mean feature values for all tracks per cluster to check general behavior patterns (BEHAV3D_UMAP_cluster_feature_heatmap.pdf)
- A barplot containing the percentages of cells belonging to each cluster per Tcell type (rows) and organoid lines (columns)

In [ ]:
# Panel for T cell analsysis
container = widgets.VBox([widgets.HTML("<i>Waiting for metadata…</i>")])
display(container)

def build_tcell_analysis_panel():
    tcell_analysis_filter = TCellAnalysisPanel(
        metadata_loader=metadata_loader, 
        )
    register_buttons_in(tcell_analysis_filter.btn_run, prefix="")
    return tcell_analysis_filter.ui  # both rendered together

render_when(container, "metadata_ready", build_tcell_analysis_panel)



## Organoid behavioral analysis

**Filtering tracks**

*Expected duration: ~1 minute*


In [ ]:
# Panel for Organoid trakc filtering
container = widgets.VBox([widgets.HTML("<i>Waiting for metadata…</i>")])
display(container)

def build_organoid_filter_panel():
    organoid_filter_panel = OrganoidFilterPanel(
        metadata_loader=metadata_loader, 
        )
    register_buttons_in(organoid_filter_panel.btn_run, prefix="")
    return organoid_filter_panel.ui  # both rendered together

render_when(container, "metadata_ready", build_organoid_filter_panel)




***Death Dynamics*** <br>

In [ ]:
# Panel for Organoid analsysis
container = widgets.VBox([widgets.HTML("<i>Waiting for metadata…</i>")])
display(container)

def build_organoid_analysis_panel():
    organoid_analysis_panel = OrganoidAnalysisPanel(
        metadata_loader = metadata_loader, 
        cell_type = "organoid"
        )
    register_buttons_in(organoid_analysis_panel.btn_run, prefix="")
    return organoid_analysis_panel.ui  # both rendered together

render_when(container, "metadata_ready", build_organoid_analysis_panel)


---

# Backproject results

---

**Visualize the cluster IDs and other features over the segments**
 <br>
 
*Duration: <15 minutes*


In [ ]:
# Panel for backprojection
container = widgets.VBox([widgets.HTML("<i>Waiting for metadata…</i>")])
display(container)

def build_backprojection_panel():
    tracking_panel = BackprojectionPanel(
        metadata_loader=metadata_loader, 
        )
    return tracking_panel.ui  # both rendered together

render_when(container, "metadata_ready", build_backprojection_panel)

